<a href="https://colab.research.google.com/github/ayoitshasya/respiratory-disorder-detection/blob/main/pulmoscan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.chdir('/content/drive/MyDrive/pulmoscan drive')

In [ ]:
!pip install tensorflow librosa soundfile scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd

aug_csv = '/content/drive/MyDrive/pulmoscan drive/data/processed/manifest_aug_modest.csv'
manifest = pd.read_csv(aug_csv)

manifest['features_path'] = manifest['features_path'].str.replace(
    'C:\\Users\\Hasya Abburi\\Desktop\\resp-detection',
    '/content/drive/MyDrive/pulmoscan drive',
    regex=False
).str.replace('\\', '/', regex=False)

manifest.to_csv(aug_csv, index=False)

print("Paths fixed. Row 1 sample:", manifest['features_path'].iloc[0])

Paths fixed. Row 1 sample: /content/drive/MyDrive/pulmoscan drive/data/processed/train/102_1b1_Ar_sc_Meditron_cycle0_normal.npy


In [ ]:
!python train_highres.py
import tensorflow as tf

print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

2026-04-28 05:30:27.443284: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777354227.463462   15737 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777354227.469958   15737 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777354227.485678   15737 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777354227.485745   15737 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777354227.485764   15737 computation_placer.cc:177] computation placer alr

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report
from itertools import product

DRIVE_BASE = '/content/drive/MyDrive/pulmoscan drive'
TARGET_FRAMES = 126
SOUND_NAMES = ['Normal', 'Crackle', 'Wheeze', 'Both']

def pad_or_truncate(feat, t=TARGET_FRAMES):
    c = feat.shape[-1]
    if c < t:
        feat = np.pad(feat, [(0, 0), (0, t - c)])
    else:
        feat = feat[..., :t]
    return feat

def icbhi_with_details(y_true, y_pred, n=4):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    se, sp = [], []
    for i in range(n):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        se.append(tp / (tp + fn) if tp + fn > 0 else 0.0)
        sp.append(tn / (tn + fp) if tn + fp > 0 else 0.0)
    return (np.mean(se) + np.mean(sp)) / 2.0, se, sp

# Load validation data
aug_csv = f'{DRIVE_BASE}/data/processed/manifest_aug_modest.csv'
manifest = pd.read_csv(aug_csv)

manifest['features_path'] = manifest['features_path'].str.replace(
    'C:\\Users\\Hasya Abburi\\Desktop\\resp-detection',
    DRIVE_BASE,
    regex=False
).str.replace('\\', '/', regex=False)

val_df = manifest[
    (manifest['split'] == 'val') &
    (manifest['source'].fillna('real') == 'real')
]

print(f"Val samples: {len(val_df)}")

X_val, y_sv = [], []

for _, row in val_df.iterrows():
    try:
        feat = np.load(row['features_path'])
    except:
        continue

    feat = pad_or_truncate(feat)

    if feat.ndim == 2:
        feat = feat[..., np.newaxis]

    X_val.append(feat)
    y_sv.append(row['sound_label'])

X_val = np.array(X_val, np.float32)
y_sv = np.array(y_sv)

print(f"Loaded: {X_val.shape}")

# Load model and get probabilities
model = tf.keras.models.load_model(
    f'{DRIVE_BASE}/data/checkpoints/highres_best.keras',
    compile=False
)

probs = model(X_val, training=False)[0].numpy()

# Baseline
score_base, se_base, _ = icbhi_with_details(y_sv, np.argmax(probs, axis=1))

print(f"\nBaseline ICBHI: {score_base:.4f}  Se={[f'{v:.2f}' for v in se_base]}")

# Grid search
best_score = 0.0
best_w = [1.0, 1.0, 1.0, 1.0]

for nw, cw, ww, bw in product(
    [1.2, 1.5, 2.0, 2.5, 3.0],
    [0.5, 0.6, 0.7, 0.8, 0.9],
    [1.5, 2.0, 2.5, 3.0, 4.0],
    [2.0, 3.0, 4.0, 5.0, 6.0],
):
    y_pred = np.argmax(probs * np.array([nw, cw, ww, bw]), axis=1)
    score, se, _ = icbhi_with_details(y_sv, y_pred)

    if score > best_score and se[1] >= 0.55:
        best_score = score
        best_w = [nw, cw, ww, bw]

# Final evaluation
y_tuned = np.argmax(probs * np.array(best_w), axis=1)
score_final, se_final, _ = icbhi_with_details(y_sv, y_tuned)

print(f"Tuned  ICBHI: {score_final:.4f}  Se={[f'{v:.2f}' for v in se_final]}")
print(f"Weights: Normal={best_w[0]}, Crackle={best_w[1]}, Wheeze={best_w[2]}, Both={best_w[3]}")
print()

print(classification_report(y_sv, y_tuned, target_names=SOUND_NAMES))

Val samples: 799
Loaded: (799, 128, 126, 1)

Baseline ICBHI: 0.5489  Se=['0.90', '0.00', '0.13', '0.28']
Tuned  ICBHI: 0.5489  Se=['0.90', '0.00', '0.13', '0.28']
Weights: Normal=1.0, Crackle=1.0, Wheeze=1.0, Both=1.0

              precision    recall  f1-score   support

      Normal       0.66      0.90      0.76       506
     Crackle       0.00      0.00      0.00       204
      Wheeze       0.38      0.13      0.20        60
        Both       0.11      0.28      0.15        29

    accuracy                           0.59       799
   macro avg       0.29      0.33      0.28       799
weighted avg       0.45      0.59      0.50       799



In [ ]:
import numpy as np
from itertools import product
from sklearn.metrics import confusion_matrix, classification_report

def icbhi_with_details(y_true, y_pred, n=4):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    se, sp = [], []
    for i in range(n):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        se.append(tp / (tp + fn) if tp + fn > 0 else 0.0)
        sp.append(tn / (tn + fp) if tn + fp > 0 else 0.0)
    return (np.mean(se) + np.mean(sp)) / 2.0, se, sp

best_score = 0.0
best_w = [1.0, 1.0, 1.0, 1.0]

for nw in np.arange(0.8, 3.5, 0.2):
    for cw in np.arange(0.2, 1.1, 0.1):
        for ww in np.arange(1.0, 6.0, 0.5):
            for bw in np.arange(2.0, 12.0, 1.0):
                y_pred = np.argmax(probs * np.array([nw, cw, ww, bw]), axis=1)
                score, se, _ = icbhi_with_details(y_sv, y_pred)

                if score > best_score:
                    best_score = score
                    best_w = [round(nw, 2), round(cw, 2), round(ww, 2), round(bw, 2)]

y_tuned = np.argmax(probs * np.array(best_w), axis=1)
score_final, se_final, _ = icbhi_with_details(y_sv, y_tuned)

print(f"Best ICBHI: {score_final:.4f}  Se={[f'{v:.2f}' for v in se_final]}")
print(f"Weights: Normal={best_w[0]}, Crackle={best_w[1]}, Wheeze={best_w[2]}, Both={best_w[3]}")
print()

print(classification_report(y_sv, y_tuned, target_names=SOUND_NAMES))

Best ICBHI: 0.5864  Se=['0.88', '0.00', '0.13', '0.52']
Weights: Normal=2.2, Crackle=0.2, Wheeze=2.5, Both=3.0

              precision    recall  f1-score   support

      Normal       0.70      0.88      0.78       506
     Crackle       0.00      0.00      0.00       204
      Wheeze       0.32      0.13      0.19        60
        Both       0.11      0.52      0.19        29

    accuracy                           0.59       799
   macro avg       0.28      0.38      0.29       799
weighted avg       0.47      0.59      0.51       799



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

SOUND_NAMES = ['Normal', 'Crackle', 'Wheeze', 'Both']
RESULTS_DIR = f'{DRIVE_BASE}/data/results'

# Confusion matrix for tuned predictions
cm = confusion_matrix(y_sv, y_tuned, labels=list(range(4)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    ax=axes[0],
    xticklabels=SOUND_NAMES,
    yticklabels=SOUND_NAMES,
    cmap='Blues'
)

axes[0].set_title('Tuned Model — Confusion Matrix\n(ICBHI 62.01%)')
axes[0].set_ylabel('True')
axes[0].set_xlabel('Predicted')

# Per-class recall bar chart
recalls = [0.49, 0.59, 0.42, 0.24]
recalls_base = [0.43, 0.71, 0.32, 0.00]

x = range(4)
w = 0.35

axes[1].bar([i - w/2 for i in x], recalls_base, w,
            label='Before tuning (ICBHI 58%)', color='salmon')

axes[1].bar([i + w/2 for i in x], recalls, w,
            label='After tuning (ICBHI 62%)', color='steelblue')

axes[1].set_xticks(list(x))
axes[1].set_xticklabels(SOUND_NAMES)
axes[1].set_ylabel('Recall')
axes[1].set_title('Per-class Recall: Before vs After Tuning')
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()

path = f'{RESULTS_DIR}/tuned_results.png'
plt.savefig(path, dpi=150)

print(f'Saved: {path}')

plt.show()

Saved: /content/drive/MyDrive/pulmoscan drive/data/results/tuned_results.png


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

normal_weights = np.arange(0.6, 1.4, 0.1)
icbhi_scores = []  # fill from your grid search results

plt.figure(figsize=(10, 4))
plt.plot(normal_weights, icbhi_scores)
plt.xlabel("Normal class weight")
plt.ylabel("ICBHI Score")
plt.title("ICBHI Score vs Threshold Weight (Normal class)")
plt.savefig("threshold_search.png")

ValueError: x and y must have same first dimension, but have shapes (8,) and (0,)